In [1]:
# --- Step 1: inspect the 311 API fields and agency volumes ---
import requests, pandas as pd

BASE = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'

row = requests.get(BASE, params={'$limit': 1}, timeout=60).json()
print('Fields:', sorted(row[0].keys()))

vol = requests.get(BASE, params={
    '$select': 'agency, count(*) AS n',
    '$where': "created_date between '2025-01-01T00:00:00' and '2025-12-31T23:59:59'",
    '$group': 'agency',
    '$order': 'n DESC',
    '$limit': 30,
}, timeout=180).json()
print('\nRequests created in 2025, by agency:')
print(pd.DataFrame(vol).to_string(index=False))

Fields: ['address_type', 'agency', 'agency_name', 'borough', 'city', 'community_board', 'complaint_type', 'created_date', 'cross_street_1', 'cross_street_2', 'descriptor', 'facility_type', 'incident_address', 'incident_zip', 'open_data_channel_type', 'park_borough', 'park_facility_name', 'police_precinct', 'resolution_action_updated_date', 'resolution_description', 'status', 'street_name', 'unique_key']

Requests created in 2025, by agency:
agency       n
  NYPD 1717594
   HPD  774535
  DSNY  326780
   DOT  212103
   DEP  197143
   DPR  110855
   DOB  103068
 DOHMH   83396
   DHS   51449
   TLC   36225
   EDC   20554
  DCWP   18278
   DOE    1737
   OOS    1111
   OTI     214


In [2]:
# --- Step 2: pull Jan 2024 - Oct 2026 for DOHMH, DHS, DPR (saved to Drive) ---
import io, os, time
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/support-ops-analytics/data_311'
os.makedirs(OUT, exist_ok=True)

# Official column list (a single JSON row omits null fields, so don't rely on it)
meta = requests.get('https://data.cityofnewyork.us/api/views/erm2-nwe9.json', timeout=60).json()
api_fields = [c['fieldName'] for c in meta['columns']]
PROB = 'problem' if 'problem' in api_fields else 'complaint_type'
DET  = 'problem_detail' if 'problem_detail' in api_fields else 'descriptor'
print('API field for Problem:', PROB, ' | for Problem Detail:', DET)

AGENCIES = ['DOHMH', 'DHS', 'DPR']
START, END = '2024-01-01T00:00:00', '2026-10-31T23:59:59'
COLS = ['unique_key', 'created_date', 'closed_date', 'due_date', 'resolution_action_updated_date',
        'agency', PROB, DET, 'status', 'open_data_channel_type',
        'incident_address', 'incident_zip', 'borough', 'latitude', 'longitude']
COLS = [c for c in COLS if c in api_fields]

CSV_URL = 'https://data.cityofnewyork.us/resource/erm2-nwe9.csv'
PAGE = 50000

def get_page(params, tries=4):
    for t in range(tries):
        try:
            r = requests.get(CSV_URL, params=params, timeout=300)
            r.raise_for_status()
            return pd.read_csv(io.StringIO(r.text), dtype=str)
        except Exception as e:
            print(f'    {type(e).__name__}, retrying in {30 * (t + 1)}s')
            time.sleep(30 * (t + 1))
    raise RuntimeError('Page failed repeatedly; rerun the cell to resume.')

for a in AGENCIES:
    path = f'{OUT}/raw_{a}.parquet'
    if os.path.exists(path):
        print(f'{a}: already downloaded, skipping'); continue
    frames, offset = [], 0
    while True:
        df = get_page({'$select': ','.join(COLS),
                       '$where': f"created_date between '{START}' and '{END}' AND agency='{a}'",
                       '$order': 'unique_key', '$limit': PAGE, '$offset': offset})
        frames.append(df)
        print(f'  {a}: {offset + len(df):,} rows')
        if len(df) < PAGE:
            break
        offset += PAGE
    pd.concat(frames, ignore_index=True).to_parquet(path, index=False)

raw = pd.concat([pd.read_parquet(f'{OUT}/raw_{a}.parquet') for a in AGENCIES], ignore_index=True)
raw = raw.rename(columns={PROB: 'problem', DET: 'problem_detail'})
for c in ['created_date', 'closed_date', 'due_date', 'resolution_action_updated_date']:
    raw[c] = pd.to_datetime(raw[c], errors='coerce')

print('\nRows by agency:'); print(raw.agency.value_counts().to_string())
print('\nCreated:', raw.created_date.min(), '->', raw.created_date.max())
print('Duplicate unique_key rows:', raw.unique_key.duplicated().sum())
print('Share with a closed date:', f'{raw.closed_date.notna().mean():.1%}')
print('Share with a due date:   ', f'{raw.due_date.notna().mean():.1%}')
print('\nStatus:'); print(raw.status.value_counts().to_string())
print('\nTop problems per agency:')
print(raw.groupby('agency').problem.value_counts().groupby(level=0).head(5).to_string())

Mounted at /content/drive
API field for Problem: complaint_type  | for Problem Detail: descriptor
  DOHMH: 50,000 rows
  DOHMH: 100,000 rows
  DOHMH: 150,000 rows
  DOHMH: 200,000 rows
  DOHMH: 247,836 rows
  DHS: 50,000 rows
  DHS: 100,000 rows
    HTTPError, retrying in 30s
    HTTPError, retrying in 60s
  DHS: 148,093 rows
    HTTPError, retrying in 30s
  DPR: 50,000 rows
    HTTPError, retrying in 30s
  DPR: 100,000 rows
  DPR: 150,000 rows
  DPR: 200,000 rows
    HTTPError, retrying in 30s
    HTTPError, retrying in 60s
  DPR: 250,000 rows
    HTTPError, retrying in 30s
    HTTPError, retrying in 60s
  DPR: 300,000 rows
  DPR: 350,000 rows
    HTTPError, retrying in 30s
    HTTPError, retrying in 60s
  DPR: 377,240 rows

Rows by agency:
agency
DPR      377240
DOHMH    247836
DHS      148093

Created: 2024-01-01 00:00:00 -> 2026-10-31 23:59:22
Duplicate unique_key rows: 0
Share with a closed date: 83.7%
Share with a due date:    0.0%

Status:
status
Closed         680393
In Progres

In [5]:
# --- Step 3: unpack the dbt project, load raw data into DuckDB, build + test ---
!pip -q install dbt-core dbt-duckdb duckdb
import os, duckdb

!unzip -o -q /content/dbt_311.zip -d /content/drive/MyDrive/support-ops-analytics/
PROJ = '/content/drive/MyDrive/support-ops-analytics/dbt_311'   # project files persist in Drive
DB   = '/content/kpis_311.duckdb'                                # rebuilt from the raw files anytime
os.environ['DUCKDB_PATH'] = DB

con = duckdb.connect(DB)
con.execute(f"create or replace table raw_311 as select * from read_parquet('{OUT}/raw_*.parquet')")
print('raw_311 rows:', con.execute('select count(*) from raw_311').fetchone()[0])
con.close()

%cd {PROJ}
!dbt build --profiles-dir .

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_311 rows: 773169
/content/drive/MyDrive/support-ops-analytics/dbt_311
21:08:49  Running with dbt=1.12.6
21:08:50  Registered adapter: duckdb=1.11.1
21:08:51  Unable to do partial parsing because saved manifest not found. Starting full parse.
21:08:55  Found 9 models, 10 data tests, 1 seed, 1 source, 511 macros
21:08:55  
21:08:55  Concurrency: 4 threads (target='dev')
21:08:55  
21:08:55  1 of 20 START sql view model main.stg_service_requests ......................... [RUN]
21:08:55  2 of 20 START seed file main.sla_targets ....................................... [RUN]
21:08:55  2 of 20 OK loaded seed file main.sla_targets ................................... [INSERT 15 in 0.33s]
21:08:55  1 of 20 OK created sql view model main.stg_service_requests .................... [OK in 0.35s]
21:08:55  6 of 20 START test not_null_stg_service_requests_request_id .................... [RUN]
21:08:55  3 of 20 START test accepted_values_stg_service_requests_agency__DOHMH__DHS__DPR  [RUN]
21:08:55 

In [6]:
# --- Step 4: read the KPIs back out of DuckDB ---
con = duckdb.connect(DB, read_only=True)
q = lambda sql: con.sql(sql).df()
pd.set_option('display.width', 200)

print('DATA QUALITY')
print(q("""
    select (select count(*) from raw_311)                     as raw_rows,
           (select count(distinct unique_key) from raw_311)   as distinct_ids,
           (select count(*) from stg_service_requests)        as clean_rows,
           (select count(*) from raw_311
             where try_cast(closed_date as timestamp) < try_cast(created_date as timestamp)) as closed_before_created
""").to_string(index=False))

print('\n1) VOLUME: average requests per complete week')
print(q("""select agency, round(avg(n)) as avg_per_week, min(n) as min_week, max(n) as max_week
           from (select agency, week_start, sum(requests_opened) n from kpi_volume_weekly group by 1, 2)
           group by 1 order by 1""").to_string(index=False))

print('\n2) TIME TO RESOLUTION (days): agency-wide, then top problems by volume')
print(q("select * from kpi_resolution_time where problem = 'ALL' order by agency").to_string(index=False))
print(q("""select * from kpi_resolution_time where problem <> 'ALL'
           qualify row_number() over (partition by agency order by requests_closed desc) <= 4
           order by agency, requests_closed desc""").to_string(index=False))

print('\n3) BACKLOG: open requests at snapshot, and the peak')
print(q("""select agency,
                  last(open_backlog order by day) as open_at_snapshot,
                  max(open_backlog)               as peak_backlog,
                  arg_max(day, open_backlog)      as peak_day
           from kpi_backlog_daily group by 1 order by 1""").to_string(index=False))

print('\n4) SLA ATTAINMENT by problem (all months), lowest first')
print(q("""select agency, problem, any_value(sla_days) as sla_days, sum(requests_evaluated) as evaluated,
                  round(sum(met_target) * 1.0 / sum(requests_evaluated), 3) as attainment
           from kpi_sla_attainment_monthly group by 1, 2
           having sum(requests_evaluated) >= 500 order by attainment""").to_string(index=False))

print('\n5-6) REPEAT AND DUPLICATE RATES (all months)')
print(q("""select r.agency,
                  round(sum(r.repeat_requests) * 1.0 / sum(r.located_requests), 3)    as repeat_rate,
                  round(sum(d.duplicate_requests) * 1.0 / sum(d.located_requests), 3) as duplicate_rate
           from kpi_repeat_rate_monthly r
           join kpi_duplicates_monthly d using (agency, month_start)
           group by 1 order by 1""").to_string(index=False))

print('\n7) WORKLOAD SHARE: average share of weekly volume, and the biggest single-week swing')
print(q("""select agency, round(avg(share_of_week), 3) as avg_share,
                  round(max(abs(share_change_wow)), 3) as max_wow_swing
           from kpi_workload_share_weekly group by 1 order by 1""").to_string(index=False))
con.close()

DATA QUALITY
 raw_rows  distinct_ids  clean_rows  closed_before_created
   773169        773169      773169                      0

1) VOLUME: average requests per complete week
agency  avg_per_week  min_week  max_week
   DHS         995.0       4.0    1831.0
 DOHMH        1676.0     905.0    2178.0
   DPR        2555.0     510.0    7354.0

2) TIME TO RESOLUTION (days): agency-wide, then top problems by volume
agency problem  requests_closed  median_days  p90_days  mean_days
   DHS     ALL           112218         0.28      8.10       3.82
 DOHMH     ALL           229772         2.61     77.31      32.36
   DPR     ALL           302098         9.93    278.90      71.04
agency                    problem  requests_closed  median_days  p90_days  mean_days
   DHS Homeless Person Assistance            73370         0.18      1.13       0.72
   DHS                 Encampment            38848         0.90     31.50       9.68
 DOHMH                     Rodent            91690         0.03    

In [7]:
# --- Step 5: three quick checks on anomalies before writing up ---
con = duckdb.connect(DB, read_only=True)
q = lambda sql: con.sql(sql).df()

print('a) DHS weeks with the lowest volume (min week was only 4 requests)')
print(q("""select week_start, sum(requests_opened) as requests
           from kpi_volume_weekly where agency = 'DHS'
           group by 1 order by 2 limit 6""").to_string(index=False))

print('\nb) Requests still open at the snapshot: how old are they?')
print(q("""select agency, count(*) as open_requests,
                  round(median(date_diff('day', created_at, timestamp '2026-10-31 23:59:59'))) as median_age_days,
                  round(avg(case when created_at < timestamp '2025-10-31' then 1.0 else 0 end), 3) as share_older_than_1yr
           from int_requests_enriched where is_open_at_snapshot
           group by 1 order by 1""").to_string(index=False))
print(q("""select agency, status, count(*) as n from int_requests_enriched
           where is_open_at_snapshot group by 1, 2 order by 1, 3 desc""").to_string(index=False))

print('\nc) Do closures cluster at exactly 60 days? (share of closed requests closed on day 59-61)')
print(q("""select agency, problem, count(*) as closed,
                  round(avg(case when resolution_hours / 24 between 59 and 61 then 1.0 else 0 end), 3) as share_closed_day_59_61
           from int_requests_enriched
           where resolution_hours is not null
             and problem in ('Food Establishment', 'Smoking or Vaping', 'Mobile Food Vendor', 'Rodent', 'Damaged Tree')
           group by 1, 2 order by share_closed_day_59_61 desc""").to_string(index=False))
con.close()

a) DHS weeks with the lowest volume (min week was only 4 requests)
week_start  requests
2026-02-02       4.0
2026-02-09     175.0
2026-02-23     269.0
2025-12-22     375.0
2025-02-17     378.0
2024-02-12     385.0

b) Requests still open at the snapshot: how old are they?
agency  open_requests  median_age_days  share_older_than_1yr
   DHS          35875            785.0                 0.935
 DOHMH          18064            151.0                 0.407
   DPR          75142            159.0                 0.356
agency      status     n
   DHS      Closed 33717
   DHS In Progress  2157
   DHS Unspecified     1
 DOHMH In Progress 16951
 DOHMH      Closed  1107
 DOHMH Unspecified     6
   DPR In Progress 73558
   DPR      Closed  1577
   DPR Unspecified     7

c) Do closures cluster at exactly 60 days? (share of closed requests closed on day 59-61)
agency            problem  closed  share_closed_day_59_61
 DOHMH Food Establishment   35399                   0.583
 DOHMH Mobile Food Vendor 

In [2]:
# --- Restart cell: mount Drive, reinstall, reload raw data, unzip project, build ---
!pip -q install dbt-core dbt-duckdb duckdb
import os, duckdb, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

OUT  = '/content/drive/MyDrive/support-ops-analytics/data_311'   # raw parquet from Step 2 (already saved)
PROJ = '/content/drive/MyDrive/support-ops-analytics/dbt_311'
DB   = '/content/kpis_311.duckdb'
os.environ['DUCKDB_PATH'] = DB

con = duckdb.connect(DB)
con.execute(f"create or replace table raw_311 as select * from read_parquet('{OUT}/raw_*.parquet')")
print('raw_311 rows:', con.execute('select count(*) from raw_311').fetchone()[0])
con.close()

!unzip -o -q /content/dbt_311.zip -d /content/drive/MyDrive/support-ops-analytics/
%cd {PROJ}
!dbt build --profiles-dir .

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Mounted at /content/drive


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

raw_311 rows: 773169
/content/drive/MyDrive/support-ops-analytics/dbt_311
02:54:57  Running with dbt=1.12.6
02:54:57  Registered adapter: duckdb=1.11.1
02:55:02  Found 9 models, 10 data tests, 1 seed, 1 source, 511 macros
02:55:02  
02:55:02  Concurrency: 4 threads (target='dev')
02:55:02  
02:55:03  2 of 20 START seed file main.sla_targets ....................................... [RUN]
02:55:03  1 of 20 START sql view model main.stg_service_requests ......................... [RUN]
02:55:04  2 of 20 OK loaded seed file main.sla_targets ................................... [INSERT 15 in 1.20s]
02:55:05  1 of 20 OK created sql view model main.stg_service_requests .................... [OK in 2.43s]
02:55:05  4 of 20 START test not_null_stg_service_requests_created_at .................... [RUN]
02:55:05  5 of 20 START test not_null_stg_service_requests_problem ....................... [RUN]
02:55:05  6 of 20 START test not_null_stg_service_requests_request_id .................... [RUN]
02:55:

In [4]:
# --- Step 4: read the KPIs back out of DuckDB ---
con = duckdb.connect(DB, read_only=True)
q = lambda sql: con.sql(sql).df()
pd.set_option('display.width', 200)

print('DATA QUALITY')
print(q("""
    select (select count(*) from raw_311)                     as raw_rows,
           (select count(distinct unique_key) from raw_311)   as distinct_ids,
           (select count(*) from stg_service_requests)        as clean_rows,
           (select count(*) from raw_311
             where try_cast(closed_date as timestamp) < try_cast(created_date as timestamp)) as closed_before_created
""").to_string(index=False))

print('\n1) VOLUME: average requests per complete week')
print(q("""select agency, round(avg(n)) as avg_per_week, min(n) as min_week, max(n) as max_week
           from (select agency, week_start, sum(requests_opened) n from kpi_volume_weekly group by 1, 2)
           group by 1 order by 1""").to_string(index=False))

print('\n2) TIME TO RESOLUTION (days): agency-wide, then top problems by volume')
print(q("select * from kpi_resolution_time where problem = 'ALL' order by agency").to_string(index=False))
print(q("""select * from kpi_resolution_time where problem <> 'ALL'
           qualify row_number() over (partition by agency order by requests_closed desc) <= 4
           order by agency, requests_closed desc""").to_string(index=False))

print('\n3) BACKLOG: open requests at snapshot, and the peak')
print(q("""select agency,
                  last(open_backlog order by day) as open_at_snapshot,
                  max(open_backlog)               as peak_backlog,
                  arg_max(day, open_backlog)      as peak_day
           from kpi_backlog_daily group by 1 order by 1""").to_string(index=False))

print('\n4) SLA ATTAINMENT by problem (all months), lowest first')
print(q("""select agency, problem, any_value(sla_days) as sla_days, sum(requests_evaluated) as evaluated,
                  round(sum(met_target) * 1.0 / sum(requests_evaluated), 3) as attainment
           from kpi_sla_attainment_monthly group by 1, 2
           having sum(requests_evaluated) >= 500 order by attainment""").to_string(index=False))

print('\n5-6) REPEAT AND DUPLICATE RATES (all months)')
print(q("""select r.agency,
                  round(sum(r.repeat_requests) * 1.0 / sum(r.located_requests), 3)    as repeat_rate,
                  round(sum(d.duplicate_requests) * 1.0 / sum(d.located_requests), 3) as duplicate_rate
           from kpi_repeat_rate_monthly r
           join kpi_duplicates_monthly d using (agency, month_start)
           group by 1 order by 1""").to_string(index=False))

print('\n7) WORKLOAD SHARE: average share of weekly volume, and the biggest single-week swing')
print(q("""select agency, round(avg(share_of_week), 3) as avg_share,
                  round(max(abs(share_change_wow)), 3) as max_wow_swing
           from kpi_workload_share_weekly group by 1 order by 1""").to_string(index=False))
con.close()

DATA QUALITY
 raw_rows  distinct_ids  clean_rows  closed_before_created
   773169        773169      773169                      0

1) VOLUME: average requests per complete week
agency  avg_per_week  min_week  max_week
   DHS         995.0       4.0    1831.0
 DOHMH        1676.0     905.0    2178.0
   DPR        2555.0     510.0    7354.0

2) TIME TO RESOLUTION (days): agency-wide, then top problems by volume
agency problem  requests_closed  median_days  p90_days  mean_days
   DHS     ALL           112218         0.28      8.10       3.82
 DOHMH     ALL           229772         2.61     77.31      32.36
   DPR     ALL           302098         9.93    278.90      71.04
agency                    problem  requests_closed  median_days  p90_days  mean_days
   DHS Homeless Person Assistance            73370         0.18      1.13       0.72
   DHS                 Encampment            38848         0.90     31.50       9.68
 DOHMH                     Rodent            91690         0.03    